# 🎙️ 日本語ウェイクワード「先生」学習ノートブック（Kaggle 版）

**livekit-wakeword-ja** を使って「先生」の ONNX モデルを学習します。

## 実行前の設定
右パネル「Session options」→「Accelerator」→ **GPU T4 x2** を選択してください。

## 概要
1. [GPU 確認](#gpu)
2. [システムパッケージ](#sys)
3. [リポジトリ・インストール](#install)
4. [設定ファイル作成（先生）](#config)
5. [データ管理設定](#storage)
6. [セットアップ](#setup)
7. [合成データ生成（チャンク）](#generate)
8. [オーグメンテーション](#augment)
9. [学習](#train)
10. [ONNX エクスポート](#export)
11. [評価](#eval)
12. [モデルのダウンロード](#download)
13. [トラブルシューティング](#troubleshoot)

## 1. GPU 確認

In [ ]:
!nvidia-smi
import sys
print(f"Python: {sys.version}")

## 2. システムパッケージ

`espeak-ng` はアドバーサリアルフレーズの音素分解に必要です。

In [ ]:
!apt-get install -y -q espeak-ng
!espeak-ng --version

## 3. リポジトリ・インストール

In [ ]:
import os

REPO_URL = "https://github.com/hazukit/livekit-wakeword-ja.git"
REPO_DIR = "/kaggle/working/livekit-wakeword-ja"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} はクローン済みです")

os.chdir(REPO_DIR)
!pwd

In [ ]:
!pip install -q -e ".[train,voxcpm]"
!pip install -q onnxscript
!livekit-wakeword --help

## 4. 設定ファイル作成（先生）

`examples/sensei_ja.yaml` をその場で書き込みます。

In [ ]:
CONFIG = "examples/sensei_ja.yaml"

content = """model_name: sensei_ja

target_phrases:
  - "先生"
  - "せんせい"

tts_backend: voxcpm
n_samples: 3000
n_samples_val: 500
n_background_samples: 500
n_background_samples_val: 100
tts_batch_size: 50

custom_negative_phrases:
  - "先生です"
  - "先生かな"
  - "先生って"
  - "せんせ"
  - "せんせー"
  - "せいせい"
  - "先制"
  - "生成"
  - "先週"
  - "せんべい"
  - "千歳"
  - "専制"

noise_scales: [0.98]
noise_scale_ws: [0.98]
length_scales: [0.75, 1.0, 1.25]
slerp_weights: [0.2, 0.35, 0.5, 0.65, 0.8]
piper_tts:
  checkpoint_relpath: piper/en-us-libritts-high.pt

voxcpm_tts:
  model_id: openbmb/VoxCPM2
  model_cache_relpath: voxcpm/VoxCPM2
  local_model_path: null
  load_denoiser: false
  cfg_values: [2.0]
  inference_timesteps_list: [10]
  voice_design_prompts:
    - A young adult woman, clear mid-pitch voice, moderate pace, calm and professional
    - A young adult man, warm baritone, steady pace, friendly and articulate
    - A middle-aged woman, slightly low pitch, measured pace, confident tone
    - A middle-aged man, deep resonant voice, slow deliberate pace
    - An older adult woman, soft gentle voice, slower pace, kind tone
    - An older adult man, gravelly tenor, moderate pace, matter-of-fact
    - A speaker with a higher pitch, enthusiastic and upbeat, medium-fast pace
    - A speaker with a lower pitch, relaxed and laid-back, slower pace
    - A news-anchor style voice, authoritative, even rhythm
    - A quiet intimate voice, close-mic feel, slow and clear
    - "日本語を自然に話す大人の女性の声"
    - "日本語を自然に話す大人の男性の声"
    - "落ち着いた日本語の話し声"
    - "少し高めの日本語の声"
    - "少し低めの日本語の声"

data_dir: /tmp/livekit-data
output_dir: ./output

augmentation:
  clip_duration: 2.0
  batch_size: 16
  rounds: 3
  background_paths: [/tmp/livekit-data/backgrounds]
  rir_paths: [/tmp/livekit-data/rirs]

model:
  model_type: conv_attention
  model_size: medium

steps: 50000
learning_rate: 0.0001
weight_decay: 0.01
label_smoothing: 0.05
max_negative_weight: 3000
target_fp_per_hour: 0.1

batch_n_per_class:
  positive: 50
  adversarial_negative: 50
  ACAV100M_sample: 1024
  background_noise: 50
"""

with open(CONFIG, "w") as f:
    f.write(content)

print(f"✅ {CONFIG} を作成しました")

## 5. データ管理設定

セッションをまたいでデータを保持するため、**Kaggle Dataset** (`sensei-ja-clips`) にバックアップします。

### 初回 / 新しいノートブックで始める場合
1. このノートブックの右パネル「Add Input」→「Your Datasets」→ **sensei-ja-clips** を追加
2. このセルを実行すると `/kaggle/input/datasets/yuriumaki/sensei-ja-clips/` から自動復元されます

### バックアップのタイミング
- チャンク generate 完了ごと（自動）
- export 完了後（自動）

In [ ]:
import os, shutil, glob, subprocess, json, zipfile

USERNAME = "yuriumaki"
DATASET = "sensei-ja-clips"
STAGING = "/kaggle/working/staging"
DATASET_INPUT = f"/kaggle/input/datasets/{USERNAME}/{DATASET}"


def backup_to_dataset():
    if not os.path.exists("output/sensei_ja"):
        print("⚠️ output/sensei_ja が見つかりません。")
        return
    os.makedirs(STAGING, exist_ok=True)
    zip_path = f"{STAGING}/sensei_ja.zip"
    if os.path.exists(zip_path):
        os.remove(zip_path)
    print("📦 zip 圧縮中...")
    shutil.make_archive(f"{STAGING}/sensei_ja", "zip", "output/sensei_ja")
    with open(f"{STAGING}/dataset-metadata.json", "w") as f:
        json.dump({"title": DATASET, "id": f"{USERNAME}/{DATASET}", "licenses": [{"name": "CC0-1.0"}]}, f)
    print("⬆️ アップロード中...")
    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", STAGING, "-m", "checkpoint", "--dir-mode", "zip"],
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
    n_pos = len(glob.glob("output/sensei_ja/positive_train/*.wav"))
    n_neg = len(glob.glob("output/sensei_ja/negative_train/*.wav"))
    print(f"✅ バックアップ完了 — positive: {n_pos}, negative: {n_neg}")


def restore_from_dataset():
    if not os.path.exists(DATASET_INPUT):
        print("ℹ️ sensei-ja-clips データセットが追加されていません。")
        print("   右パネル「Add Input」→「Your Datasets」→ sensei-ja-clips を追加してください。")
        return
    if os.path.exists("output/sensei_ja"):
        print("ℹ️ output/sensei_ja はすでに存在します。")
        return
    print("📂 復元中...")
    os.makedirs("output/sensei_ja", exist_ok=True)
    zip_path = f"{DATASET_INPUT}/sensei_ja.zip"
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall("output/sensei_ja")
    else:
        shutil.copytree(DATASET_INPUT, "output/sensei_ja", dirs_exist_ok=True)
    n_pos = len(glob.glob("output/sensei_ja/positive_train/*.wav"))
    n_neg = len(glob.glob("output/sensei_ja/negative_train/*.wav"))
    print(f"✅ 復元完了 — positive: {n_pos}, negative: {n_neg}")


restore_from_dataset()

## 6. セットアップ

VoxCPM2 モデル・背景音声・RIR をダウンロードします。

> **注意**: VoxCPM2 は数 GB あります。初回は 10〜30 分かかります。

In [ ]:
!livekit-wakeword setup --config {CONFIG}

## 7. 合成データ生成（チャンク実行）

VoxCPM2 で「先生」「せんせい」のポジティブクリップとアドバーサリアルネガティブクリップを生成します。

- `CHUNK_SIZE=1000` 件ずつ生成 → 完了ごとに Kaggle Dataset にバックアップ
- セッションが切れても再実行するだけで続きから再開します

> **⚠️ TODO**: VoxCPM2 の日本語発音品質はエンドツーエンドで未検証です。生成後に再生して確認してください。

In [ ]:
import yaml

TARGET_N_SAMPLES = 3000
CHUNK_SIZE = 1000

with open(CONFIG) as f:
    base_cfg = yaml.safe_load(f)

n = min(CHUNK_SIZE, TARGET_N_SAMPLES)
while True:
    cfg = dict(base_cfg)
    cfg["n_samples"] = n
    cfg["n_background_samples"] = 0
    chunk_path = "examples/sensei_ja_chunk.yaml"
    with open(chunk_path, "w") as f:
        yaml.dump(cfg, f, allow_unicode=True)

    print(f"\n=== positive/negative: n_samples={n} まで生成中 ===")
    !livekit-wakeword generate {chunk_path}
    backup_to_dataset()

    if n >= TARGET_N_SAMPLES:
        break
    n = min(n + CHUNK_SIZE, TARGET_N_SAMPLES)

print("\n✅ ポジティブ / ネガティブ生成完了")

In [ ]:
TARGET_N_BACKGROUND = 500
BG_CHUNK_SIZE = 500

n_bg = min(BG_CHUNK_SIZE, TARGET_N_BACKGROUND)
while True:
    cfg = dict(base_cfg)
    cfg["n_samples"] = TARGET_N_SAMPLES
    cfg["n_background_samples"] = n_bg
    chunk_path = "examples/sensei_ja_chunk.yaml"
    with open(chunk_path, "w") as f:
        yaml.dump(cfg, f, allow_unicode=True)

    print(f"\n=== background: n_background_samples={n_bg} まで生成中 ===")
    !livekit-wakeword generate {chunk_path}
    backup_to_dataset()

    if n_bg >= TARGET_N_BACKGROUND:
        break
    n_bg = min(n_bg + BG_CHUNK_SIZE, TARGET_N_BACKGROUND)

print("\n✅ 背景ノイズ生成完了")

In [ ]:
import glob, os
output_dir = "output/sensei_ja"
for subdir in ["positive_train", "negative_train", "background_train"]:
    files = glob.glob(f"{output_dir}/{subdir}/*.wav")
    print(f"{subdir}: {len(files)} クリップ")

In [ ]:
import glob, random
from IPython.display import Audio, display

files = sorted(glob.glob("output/sensei_ja/positive_train/*.wav"))
if files:
    for f in random.sample(files, min(3, len(files))):
        print(f)
        display(Audio(f))
else:
    print("クリップが見つかりません。generate を先に実行してください。")

## 8. オーグメンテーション

In [ ]:
!livekit-wakeword augment {CONFIG}

## 9. 学習

- デフォルト: `medium` サイズ、`50,000` ステップ
- Kaggle T4 で概算 **1〜3 時間**
- チェックポイントは `output/sensei_ja/` に保存

In [ ]:
!livekit-wakeword train {CONFIG}

## 10. ONNX エクスポート

In [ ]:
!livekit-wakeword export {CONFIG}
backup_to_dataset()  # ONNX + チェックポイントをバックアップ

In [ ]:
import os
onnx_path = "output/sensei_ja/sensei_ja.onnx"
if os.path.exists(onnx_path):
    size_mb = os.path.getsize(onnx_path) / 1024 / 1024
    print(f"✅ ONNX エクスポート成功: {onnx_path} ({size_mb:.1f} MB)")
else:
    print(f"❌ {onnx_path} が見つかりません。")

## 11. 評価

| 指標 | 目標 |
|---|---|
| FPPH | < 0.2 |
| Recall | ≥ 80% |
| AUT | < 0.1 |

In [ ]:
!livekit-wakeword eval {CONFIG}

## 12. モデルのダウンロード

Kaggle では右パネルの「Output」タブからもダウンロードできます。

In [ ]:
import shutil, os

onnx_path = "output/sensei_ja/sensei_ja.onnx"
if os.path.exists(onnx_path):
    shutil.copy(onnx_path, "/kaggle/working/sensei_ja.onnx")
    print("✅ /kaggle/working/sensei_ja.onnx にコピーしました")
    print("右パネルの Output タブからダウンロードしてください。")
else:
    print("❌ ONNX ファイルが見つかりません。export を先に実行してください。")

## 13. トラブルシューティング

---

### セッションが切れた

上から順に再実行するだけです。`restore_from_dataset()` が前回のデータを復元し、チャンク生成ループが続きから再開します。

---

### GPU が割り当てられない

右パネル「Session options」→「Accelerator」→ **GPU T4 x2** を選択してください。

---

### `No module named 'voxcpm'`

```bash
!pip install -e ".[train,voxcpm]"
```

を再実行してください。

---

### `ModuleNotFoundError: No module named 'onnxscript'`

```bash
!pip install onnxscript
```

を実行してから export を再実行してください。

---

### VoxCPM2 が日本語を正しく発音しない

> **⚠️ TODO**: VoxCPM2 の日本語合成はエンドツーエンドで未検証です。

セル 7 の再生セルで確認してください。不自然な場合は `target_phrases` を `["せんせい"]`（ひらがなのみ）に絞って試してください。

---

### 学習が遅い

`sensei_ja.yaml` の `steps` を 30,000 に、`model_size` を `small` に変更して試してください。